# La logística de la arena

Cinco millones de toneladas de arena por año viajan 1.461 km en camión, de
Ibicuy a Añelo, porque la arena cercana no aguantó la presión y el tren no
llega. Este notebook mide esa cadena con los volúmenes reales año a año, la
compara con las alternativas que están sobre la mesa, barcaza y camión,
barcaza y tren, hidrovía por el río Negro, arena cercana, con las mismas
reglas para todas, y pone los escenarios de demanda que circulan: 5, 8 y 15
millones de toneladas. Después mira cómo lo resolvieron otros países. Todo con
datos abiertos y costos publicados en 2026, declarados uno por uno.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, json
sys.path.insert(0, "../src")
import geopandas as gpd, pandas as pd, numpy as np
from IPython.display import IFrame, display

from alab import DATA_RAW, DATA_PROC, DOCS
from alab.demand import read_fracture
from alab.route import sand_route, waypoints_frame
from alab.animation import SUPUESTOS, yearly_series
from alab.logistics import CHAINS, EMISIONES_G_TKM, MODES, BENCHMARKS, chains_table, scenarios
from alab.animation2 import chain_paths, logistics_animation_html, CONTEXTO
from alab.roads import route_segments, segments_geojson
from shapely.geometry import LineString
pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 70)

## 1. La cadena de hoy, medida

Los volúmenes salen del registro de fractura: toneladas de arena nacional e
importada por año en los pozos no convencionales de la cuenca Neuquina. Los
supuestos para pasar de toneladas a camiones y a dólares están en un solo
lugar y se muestran.

In [2]:
route = sand_route(cache=DATA_RAW / "ruta_arena_osrm.json"); km = float(route.iloc[0]["km"])
frac = read_fracture(DATA_RAW / "fractura_adjunto_iv.csv")
y = yearly_series(frac, route_km=km); y = y[y.anio <= 2026]
print(f"ruta de {km:,.0f} km · supuestos: {SUPUESTOS}")
y[["anio", "nat_t", "imp_t", "viajes", "viajes_dia", "camiones", "flete_musd", "usd_km_anio", "flete_acum_musd"]].assign(nat_t=lambda d: (d.nat_t/1e3).round(0), imp_t=lambda d: (d.imp_t/1e3).round(0)).rename(columns={"nat_t": "nacional_kt", "imp_t": "importada_kt"}).set_index("anio")

ruta de 1,461 km · supuestos: {'t_por_camion': 30.0, 'dias_operativos': 300, 'dias_ciclo': 6.0, 'usd_flete_por_t': 70.0, 'camiones_por_punto': 25}


,nacional_kt,importada_kt,viajes,viajes_dia,camiones,flete_musd,usd_km_anio,flete_acum_musd
anio,,,,,,,,
2012,4.0,22.0,133.0,0.4,2.0,0.3,192.0,0.3
2013,21.0,103.0,715.0,2.4,14.0,1.5,1027.0,1.8
2014,68.0,174.0,2274.0,7.6,46.0,4.8,3268.0,6.6
2015,73.0,235.0,2434.0,8.1,49.0,5.1,3499.0,11.7
2016,300.0,142.0,9984.0,33.3,200.0,21.0,14350.0,32.7
2017,586.0,132.0,19539.0,65.1,391.0,41.0,28085.0,73.7
2018,934.0,176.0,31133.0,103.8,623.0,65.4,44750.0,139.1
2019,1253.0,265.0,41762.0,139.2,835.0,87.7,60028.0,226.8
2020,704.0,38.0,23478.0,78.3,470.0,49.3,33746.0,276.1


En 2025 son 168.000 viajes cargados, 561 por día, unos 3.400 camiones en
circulación contra los 4.300 de flota que cuenta la prensa, 354 millones de
dólares de flete a precios de 2026 y 242.000 dólares por cada kilómetro de
ruta. El acumulado desde 2012 pasa los 1.400 millones.

## 2. Las alternativas, con las mismas reglas

Cada cadena es una lista de tramos con su modo y sus kilómetros, más un costo
por tonelada puesta en el pozo. Donde la fuente da el costo de la cadena
completa se usa ese número: el estudio que cita El Cronista para las cadenas
por Bahía Blanca, GlobalPorts para la hidrovía. El camión de hoy son los 70
dólares de flete más los 27 de última milla que desglosa Infobae. Las
emisiones son los factores europeos de 2018 pozo a rueda, los únicos con
método comparable para los cuatro modos.

In [3]:
print("costos unitarios de referencia:"); [print(f"  {m:9s} {v['usd_tkm']:.4f} USD/tkm  ({v['fuente']})") for m, v in MODES.items()]
print("emisiones gCO2e/tkm:", EMISIONES_G_TKM)
chains_table()[["cadena", "usd_t_pozo", "km_total", "km_camion", "kg_co2e_t", "estado", "limite"]]

costos unitarios de referencia:
  camion    0.0479 USD/tkm  (flete Ibicuy-Añelo 70 USD/t (Infobae, 15/8/2026) sobre 1.461 km)
  fluvial   0.0132 USD/tkm  (Ibicuy-Bahía Blanca por agua 16,9 USD/t sobre 1.279 km (El Cronista, 4/9/2026))
  maritimo  0.0132 USD/tkm  (mismo tramo por agua; la fuente no separa río de mar)
  tren      0.0239 USD/tkm  (derivado: 35,2 total de agua+tren+camión menos 16,9 de agua y 2,4 de última milla (El Cronista, 4/9/2026))
emisiones gCO2e/tkm: {'camion': 137.0, 'tren': 24.0, 'fluvial': 33.4, 'maritimo': 6.6}


,cadena,usd_t_pozo,km_total,km_camion,kg_co2e_t,estado,limite
0,"camión directo, hoy",97.0,1461,1461,200.2,"existe: 4.300 camiones, 70 a 75 h por tramo",rutas al límite; 700 a 800 camiones más por año si la actividad se...
1,barcaza a Bahía Blanca y camión,62.7,1899,620,100.1,"en construcción: terminal de PTP en Ibicuy, 12 MUSD, permiso 2024",falta terminal de arena en Bahía Blanca; 620 km de camión por la R...
2,"barcaza, Tren Norpatagónico y camión",35.2,1994,50,38.0,"no existe: el tren mueve <100 kt/año, 37 % de la vía en buen estad...","500 MUSD y una licitación que no está; 1,5 Mt el primer año, 6 Mt ..."
3,hidrovía patagónica por el río Negro,48.0,2350,50,48.0,"en estudio: dos informes técnicos, el tercero con inversiones pend...","dragado, terminales, reforma del cabotaje; el puerto de San Antoni..."
4,arena cercana de Neuquén,29.9,60,60,8.2,"en prueba: <20.000 t por mes, YPF y Vista",calidad por confirmar en pozo; volumen chico; sin ensayos públicos


La lectura es directa: el camión de hoy cuesta 97 dólares por tonelada hasta
el pozo y emite 200 kg de CO2e por tonelada. La cadena por barcaza y tren
costaría 35 y emitiría 38. La diferencia es que la primera existe y la segunda
no: el tren mueve menos de 100.000 toneladas por año, tiene el 37 % de la vía
en buen estado y le faltan 83 km hasta Añelo.

## 3. Escenarios: 5, 8 y 15 millones de toneladas

Las tres cifras que circulan: lo que se bombeó en 2025, lo que se proyecta
para 2027 y el techo que se menciona para la década. Para cada cadena, el
costo anual, el ahorro contra el camión, los camiones de larga distancia que
quedan en la ruta, las emisiones, y para el tren, en cuántos años el ahorro
paga los 500 millones. Dos versiones del ahorro: si toda la demanda fuera por
esa cadena, y con la capacidad que la cadena tiene hoy o al inicio.

In [4]:
sc = scenarios()
sc[["demanda_mt", "cadena", "costo_musd", "ahorro_musd", "ahorro_con_capacidad_musd", "camiones_larga_distancia", "co2_kt", "repago_tren_anios", "repago_tren_con_capacidad_anios", "cubre_pct"]]

,demanda_mt,cadena,costo_musd,ahorro_musd,ahorro_con_capacidad_musd,camiones_larga_distancia,co2_kt,repago_tren_anios,repago_tren_con_capacidad_anios,cubre_pct
0,5.0,"camión directo, hoy",485.0,0.0,0.0,3333,1000.8,NaN,NaN,NaN
1,5.0,barcaza a Bahía Blanca y camión,313.5,171.5,171.5,1415,500.4,NaN,NaN,NaN
2,5.0,"barcaza, Tren Norpatagónico y camión",176.0,309.0,92.7,0,189.8,1.6,5.4,30.0
3,5.0,hidrovía patagónica por el río Negro,240.0,245.0,245.0,0,240.1,NaN,NaN,NaN
4,5.0,arena cercana de Neuquén,149.4,335.6,16.1,0,41.1,NaN,NaN,5.0
5,8.0,"camión directo, hoy",776.0,0.0,0.0,5333,1601.3,NaN,NaN,NaN
6,8.0,barcaza a Bahía Blanca y camión,501.6,274.4,274.4,2263,800.7,NaN,NaN,NaN
7,8.0,"barcaza, Tren Norpatagónico y camión",281.6,494.4,92.7,0,303.6,1.0,5.4,19.0
8,8.0,hidrovía patagónica por el río Negro,384.0,392.0,392.0,0,384.2,NaN,NaN,NaN
9,8.0,arena cercana de Neuquén,239.0,537.0,16.1,0,65.8,NaN,NaN,3.0


Con 5 millones de toneladas, el tren ahorraría 309 millones por año si
pudiera llevarlo todo, y pagaría su inversión en menos de dos años. Con la
capacidad de 1,5 millones que se le atribuye al primer año, el ahorro es de 93
millones y el repago pasa a cinco años y medio. La hidrovía patagónica está en
el medio y no tiene ni capacidad conocida ni inversión estimada. La arena
cercana es la más barata por tonelada y la que menos emite, y hoy mueve
20.000 toneladas por mes, el 5 % de la demanda, con la calidad por confirmar.

## 4. Cómo lo resolvieron otros

In [5]:
BENCHMARKS

,pais,arena,distancia_km,modo,logistica_pct,que_cambio,fuente
0,"Estados Unidos, hasta 2017","Northern White, Wisconsin",2000,tren unitario y camión,75 %,apareció arena regional en el Permian; el flete era la mitad del c...,PLG Consulting 2018; AOGR 12/2017
1,"Estados Unidos, hoy",arena de duna del Permian,50,"camión, cajas y silos; cinta de 68 km",la menor,60 % menos por tonelada; Dune Express de 400 MUSD mueve 13 Mt/año ...,JPT 12/2018; JPT 2024
2,Canadá,Wisconsin y Peace River,1500,tren a terminales y camión; arena local sin tren,sin dato,mezcla de importada por tren y local cercana,Source Energy Services
3,Rusia,cerámica de los Urales,2000,"tren a depósitos en Siberia, camión y caminos de invierno",sin dato,sin arena apta: 99 % cerámica; depósitos cargados antes del invierno,ROGTEC
4,"Argentina, hoy","Ibicuy, Entre Ríos",1461,camión,más del 70 %,la arena cercana perdió por calidad; el tren no llega y el río est...,Infobae 8/2026; este repo


Estados Unidos pasó por las dos etapas en diez años: arena de calidad a 2.000
km en tren, con la logística en tres cuartos del precio, y después arena
regional a 50 km, con la cinta transportadora como última vuelta de tuerca.
Argentina está en la primera etapa con el modo equivocado: la misma
proporción de logística en el precio, pero en camión.

## 5. La ruta animada, versión 2

Los blancos de arena debajo de la traza, el ferrocarril con su estado, los
ríos y los puertos como capas, y un selector de cadena que cambia las
unidades que circulan, el costo del año y las emisiones. Cada tramo de ruta
nacional se pinta según cuánto pesan los camiones de arena en el tránsito
que tenía en 2017, la medida del notebook 03. Al final, una pantalla con lo
que habría costado cada camino desde 2012.

In [6]:
cp = chain_paths(DATA_RAW / "ruta_arena_osrm.json", DATA_RAW / "ruta_bahiablanca_anelo_osrm.json")
layers = {k: json.load(open(DATA_PROC / f"anim_{k}.geojson", encoding="utf-8")) for k in ("blancos", "ferrocarril", "rios")}
# tramos de ruta nacional con tránsito de 2017, para pintar la presión de los camiones año a año (ver notebook 03)
tmda = gpd.read_file(DATA_RAW / "tmda_rutas_nacionales.geojson")
bb = json.loads((DATA_RAW / "ruta_bahiablanca_anelo_osrm.json").read_text(encoding="utf-8"))
route_bb = gpd.GeoDataFrame({"km": [round(bb["routes"][0]["distance"] / 1000)]}, geometry=[LineString(bb["routes"][0]["geometry"]["coordinates"])], crs="EPSG:4326")
seg = gpd.GeoDataFrame(pd.concat([route_segments(tmda, route, "Ibicuy"), route_segments(tmda, route_bb, "Bahía Blanca")], ignore_index=True), crs="EPSG:4326")
layers["tramos"] = segments_geojson(seg); json.dump(layers["tramos"], open(DATA_PROC / "anim_tramos.geojson", "w", encoding="utf-8"), ensure_ascii=False)
pt = gpd.read_file(DATA_PROC / "anim_puertos.geojson")
layers["puertos"] = [{"nombre": r["nombre"].title(), "lat": float(r.geometry.y), "lon": float(r.geometry.x)} for _, r in pt.iterrows()]
out = logistics_animation_html(y, cp, layers, waypoints_frame(), DOCS / "ruta_animada.html", partial_year=2026)
sc.to_csv(DATA_PROC / "escenarios.csv", index=False); chains_table().to_csv(DATA_PROC / "cadenas.csv", index=False)
IFrame("../docs/ruta_animada.html", width="100%", height=620)

## Límites

- Los costos por tonelada vienen de tres fuentes distintas con bases no
  idénticas; se declara cuál es cuál. Las cadenas por agua usan estudios que
  todavía no tienen la inversión estimada.
- Los factores de emisión son europeos de 2018. La flota argentina es más
  vieja y las rutas peores: el camión probablemente emite más que 137 g/tkm.
- Las trazas de barcaza y tren son puntos de paso aproximados, no rutas
  náuticas ni ferroviarias exactas.
- Se supone toda la arena nacional por una sola cadena. En la realidad
  conviven, y hasta 2020 una parte venía de Río Negro y Chubut.